# Moreno-Louzada's design (ADM2 panel)

Do gold deposits upstream, interacted with the gold price, raise congenital anomalies and
worsen infant health? This notebook replicates and extends the design of Moreno-Louzada's
thesis (recorded as Evidence E9 in
[`output/research/mining_economics_leads.md`](../../research/mining_economics_leads.md)):

- **Exposure:** any SGB gold occurrence in an Otto level-4 sub-basin upstream of one the
  municipality intersects, within the same level-3 basin (`gold_upstream_otto4`, from
  `geology_adm2_upstream`), times the log annual gold price in BRL (Pink Sheet × BCB).
- **Sample:** the Legal Amazon (eight states and Maranhão west of 44°W), without the
  municipalities holding a gold occurrence themselves, 2007–2021.
- **Model:** OLS of the outcome rate on the interaction, municipality and year FE,
  weighted by the municipality's mean births.
- **Outcomes:** per 1,000 births: congenital anomalies at birth, low birth weight (TabNet
  has no mean birthweight), preterm (from 2012), APGAR at 5 minutes below 6, infant deaths
  excluding external causes (ICD-10 V–Y), infant deaths from congenital malformations, and
  SIH congenital-anomaly hospitalizations (ICD-10 chapter XVII); the placebo is injury
  hospitalizations (chapter XIX) per 1,000 population.
- **Inference:** Otto-3 clustered SEs; the restricted wild-cluster bootstrap on Otto-3
  basins (9,999 draws); two-way clustering by basin and year; the price-exposure (P-E)
  variance of Moreno-Louzada, Figueira and Picchetti (arXiv 2512.10076); and, for the
  baseline, randomization inference over 1,999 random-walk price paths.

The estimators live in `src/analysis/moreno_louzada.py`. Tables are written to
`output/analysis/health/moreno_louzada/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import warnings

import numpy as np
import pandas as pd

from src.analysis import moreno_louzada as M

warnings.filterwarnings("ignore", message=r".*singleton fixed effect")
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda value: f"{value:.4g}")

OUT = PROJECT_ROOT / "output" / "analysis" / "health" / "moreno_louzada"
OUT.mkdir(parents=True, exist_ok=True)
COLUMNS = ["estimate", "se_basin", "p_basin", "p_wcb", "se_two_way", "p_two_way", "se_pe", "p_pe",
           "n_obs", "n_municipalities", "n_exposed", "n_basins", "mean_y"]

frame = M.build_frame()
print(f"{len(frame):,} municipality-years · {frame.mun_id.nunique():,} municipalities · {frame.year.min()}–{frame.year.max()}")

15,440 municipality-years · 772 municipalities · 2004–2023


## 1. Sample and exposure

Moreno-Louzada's sample has 768 Legal Amazon municipalities and 616 without a deposit of
their own.

In [2]:
sample = M.sample_table(frame)
sample.to_csv(OUT / "sample.csv", header=["municipalities"])
display(sample.to_frame("municipalities"))

in_sample = frame[frame["sample"]].drop_duplicates("mun_id")
by_state = in_sample.groupby("state")[M.EXPOSURE].agg(["size", "sum"]).rename(columns={"size": "municipalities", "sum": "exposed"})
display(by_state.T)

prices = frame.drop_duplicates("year").set_index("year")[["gold_usd_oz", "gold_brl_oz", M.PRICE_USD, M.PRICE]].sort_index()
display(prices.loc[2004:2023].T.round(2))
print("SD of yearly changes, 2007-21: BRL", round(prices.loc[2007:2021, M.PRICE].diff().std(), 3),
      "USD", round(prices.loc[2007:2021, M.PRICE_USD].diff().std(), 3))

,municipalities
Legal Amazon municipalities in the panel,772
with a gold occurrence of their own (dropped),154
sample,618
exposed: gold upstream (Otto-4),161
gold downstream (Otto-4),147
tin upstream (Otto-4),70
never-exploited gold upstream (Otto-4),103
Otto-3 basins in the sample,117


state,11,12,13,14,15,16,17,21,51
municipalities,22,21,52,5,110,5,124,173,106
exposed,8,2,27,3,41,4,49,4,23


year,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
gold_usd_oz,409.2,444.8,604.4,696.7,871.8,973,1225,1569,1670,1412,1266,1161,1249,1258,1269,1393,1770,1800,1801,1943
gold_brl_oz,1195,1080,1315,1351,1588,1929,2152,2633,3262,3029,2974,3848,4345,4014,4625,5504,9173,9705,9289,9700
log_gold_usd,6.01,6.1,6.4,6.55,6.77,6.88,7.11,7.36,7.42,7.25,7.14,7.06,7.13,7.14,7.15,7.24,7.48,7.5,7.5,7.57
log_gold_brl,7.09,6.98,7.18,7.21,7.37,7.56,7.67,7.88,8.09,8.02,8,8.26,8.38,8.3,8.44,8.61,9.12,9.18,9.14,9.18


SD of yearly changes, 2007-21: BRL 0.15 USD 0.135


**Reading the results.** The Legal Amazon has 772 municipalities in the panel; 154 hold a gold
occurrence themselves, leaving 618 (Moreno-Louzada: 616). 161 have gold in an upstream
Otto-4 sub-basin, spread over 117 Otto-3 basins and concentrated in Tocantins, Pará,
Mato Grosso and Amazonas; only 4 of Maranhão's 173 are exposed. The BRL gold price rises
almost monotonically, with one large step: +0.51 log points in 2020, as the real fell
and gold rose. Over 2007–2021 the interaction is therefore close to exposure × trend, with
much of its variation in the COVID years.

## 2. Replication baseline

Every outcome under the baseline: the interaction coefficient (outcome units per log
point of the BRL gold price, for an exposed municipality), with the four inferences and
the random-walk randomization p-value. Moreno-Louzada's anomaly estimate is +2.28
(SE 1.16) with the USD price.

In [3]:
baseline = M.run_table(frame, list(M.OUTCOMES), {"baseline": {}})
baseline.to_csv(OUT / "baseline.csv", index=False)
display(baseline.set_index("outcome")[COLUMNS[:8] + ["p_random_walk"] + COLUMNS[8:]])

,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,p_random_walk,n_obs,n_municipalities,n_exposed,n_basins,mean_y
outcome,,,,,,,,,,,,,,
anomaly,0.8396,0.9788,0.3928,0.4255,0.9638,0.3984,0.2541,0.0009531,0.7865,9264,618,161,117,5.256
low_birth_weight,-1.241,1.365,0.3651,0.4623,1.355,0.3755,0.5747,0.03087,0.6805,9263,618,161,117,75.3
preterm,5.539,4.167,0.1864,0.2969,3.852,0.1844,2.695,0.03984,0.718,6178,618,161,117,122
apgar5_below_6,-0.2602,0.8819,0.7684,0.7978,1.011,0.8007,0.7277,0.7206,0.8445,9264,618,161,117,7.605
infant_deaths_nonexternal,0.1182,0.3887,0.7616,0.7681,0.3155,0.7135,0.1642,0.4717,0.7195,9264,618,161,117,15.42
infant_deaths_congenital,0.3144,0.103,0.002815,0.0159,0.07956,0.001446,0.06356,7.547e-07,0.7825,9264,618,161,117,2.78
congenital_hospitalizations,-2.581,0.9756,0.009296,0.05291,1.009,0.02282,0.8399,0.002122,0.8435,9264,618,161,117,16.72
injury_hospitalizations,-0.7521,0.3134,0.01798,0.07001,0.3199,0.03392,0.1444,1.912e-07,0.824,9264,618,161,117,5.11


**Reading the results.** Moreno-Louzada's anomaly result does not replicate: +0.84 anomalies
per 1,000 births per log point (Otto-3 SE 0.98, WCB p 0.43), against his +2.28 (SE 1.16),
on a mean of 5.3. Low birth weight, preterm, APGAR and non-external infant deaths are
null under every clustered inference. Infant deaths from congenital malformations are the
exception: +0.31 per 1,000 births (mean 2.8; WCB p 0.016, two-way p 0.001). But the
placebo fails: injury hospitalizations *fall* (−0.75 per 1,000 population, Otto-3 p 0.018,
WCB p 0.07), and congenital hospitalizations fall too (−2.6, WCB p 0.053), opposite in
sign to congenital deaths. The P-E p-values reject for most outcomes, and the random-walk
p-values reject none (0.68–0.84).

## 3. Robustness and placebos

The baseline against: state×year FE; dropping 2015–17; unweighted; without Manaus and
Belém; 2004–2023; the USD price; never-exploited occurrences only (Otto-4 binary, and
log(1 + count) within 200 km and without limit along the river network); and two
placebos added to the baseline exposure, gold downstream (Otto-4) and tin upstream
(Otto-4).

In [4]:
specs = M.robustness_specs()
robustness = M.run_table(frame, list(M.OUTCOMES), {name: kwargs for name, kwargs in specs.items() if name != "baseline"},
                         randomization_specs=())
robustness = pd.concat([baseline.drop(columns="p_random_walk"), robustness], ignore_index=True)
robustness.to_csv(OUT / "robustness.csv", index=False)

for outcome in M.OUTCOMES:
    print(f"\n{outcome}")
    display(robustness[robustness.outcome == outcome].set_index(["spec", "exposure"])[COLUMNS[:8] + ["n_obs", "n_exposed"]])


anomaly


,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_exposed
spec,exposure,,,,,,,,,,
baseline,gold_upstream_otto4,0.8396,0.9788,0.3928,0.4255,0.9638,0.3984,0.2541,0.0009531,9264,161
state x year FE,gold_upstream_otto4,0.437,0.4005,0.2775,0.3069,0.3747,0.263,0.1102,7.344e-05,9264,161
drop 2015-17,gold_upstream_otto4,0.9612,0.9619,0.3197,0.3522,0.9395,0.3282,0.2143,7.244e-06,7410,161
unweighted,gold_upstream_otto4,0.4102,0.4849,0.3994,0.4191,0.5494,0.4677,0.3727,0.2712,9264,161
without Manaus and Belém,gold_upstream_otto4,0.6258,0.8844,0.4806,0.5627,0.875,0.4863,0.2839,0.0275,9234,160
2004-2023,gold_upstream_otto4,0.9862,1.037,0.3436,0.3773,1.05,0.3593,0.3292,0.002734,12347,161
USD price,gold_upstream_otto4,1.033,1.262,0.4148,0.4539,1.184,0.3975,0.5787,0.07424,9264,161
"never-exploited gold, Otto-4",gold_unexploited_upstream_otto4,2.358,1.389,0.09208,0.1125,1.378,0.1091,0.4604,3.015e-07,9264,103
"never-exploited gold within 200 km, log(1+count)",log1p_gold_unexploited_upstream_200,0.7365,0.5891,0.2137,0.1495,0.5782,0.2235,0.1767,3.063e-05,9264,58



low_birth_weight


,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_exposed
spec,exposure,,,,,,,,,,
baseline,gold_upstream_otto4,-1.241,1.365,0.3651,0.4623,1.355,0.3755,0.5747,0.03087,9263,161
state x year FE,gold_upstream_otto4,-1.743,1.253,0.1668,0.2796,1.201,0.1689,0.4574,0.000139,9263,161
drop 2015-17,gold_upstream_otto4,-1.427,1.286,0.2696,0.3866,1.219,0.2665,0.4481,0.001456,7409,161
unweighted,gold_upstream_otto4,0.7388,1.309,0.5735,0.5817,1.185,0.5429,0.7646,0.3339,9263,161
without Manaus and Belém,gold_upstream_otto4,-0.6846,1.312,0.6029,0.6269,1.374,0.626,0.7373,0.3532,9233,160
2004-2023,gold_upstream_otto4,-0.8992,1.137,0.4305,0.5763,1.133,0.4372,0.4141,0.0299,12346,161
USD price,gold_upstream_otto4,-2.09,1.999,0.2981,0.362,2.045,0.3243,1.462,0.1529,9263,161
"never-exploited gold, Otto-4",gold_unexploited_upstream_otto4,0.71,2.053,0.7301,0.7598,2.031,0.7318,0.7977,0.3734,9263,103
"never-exploited gold within 200 km, log(1+count)",log1p_gold_unexploited_upstream_200,0.1916,1.052,0.8558,0.7925,1.041,0.8566,0.3355,0.5679,9263,58



preterm


...

,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_exposed
spec,exposure,,,,,,,,,,
baseline,gold_upstream_otto4,0.3144,0.103,0.002815,0.0159,0.07956,0.001446,0.06356,7.547e-07,9264,161
state x year FE,gold_upstream_otto4,0.1466,0.1009,0.149,0.1527,0.09249,0.1354,0.09517,0.1236,9264,161
drop 2015-17,gold_upstream_otto4,0.311,0.1037,0.003308,0.0227,0.08357,0.003372,0.0612,3.729e-07,7410,161
unweighted,gold_upstream_otto4,0.0687,0.1874,0.7145,0.7164,0.1545,0.6633,0.1318,0.6022,9264,161
without Manaus and Belém,gold_upstream_otto4,0.3114,0.1224,0.01229,0.0276,0.1137,0.01602,0.08222,0.0001525,9234,160
2004-2023,gold_upstream_otto4,0.1791,0.09491,0.06162,0.05831,0.09005,0.06129,0.05745,0.00182,12347,161
USD price,gold_upstream_otto4,0.4769,0.2101,0.02505,0.0483,0.1647,0.01173,0.1529,0.001816,9264,161
"never-exploited gold, Otto-4",gold_unexploited_upstream_otto4,0.2387,0.151,0.1168,0.166,0.1785,0.2025,0.1428,0.09459,9264,103
"never-exploited gold within 200 km, log(1+count)",log1p_gold_unexploited_upstream_200,0.05457,0.06421,0.3971,0.437,0.06649,0.4255,0.05587,0.3287,9264,58



congenital_hospitalizations


,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_exposed
spec,exposure,,,,,,,,,,
baseline,gold_upstream_otto4,-2.581,0.9756,0.009296,0.05291,1.009,0.02282,0.8399,0.002122,9264,161
state x year FE,gold_upstream_otto4,-0.9753,0.5621,0.08539,0.1353,0.6559,0.1592,0.7209,0.1761,9264,161
drop 2015-17,gold_upstream_otto4,-2.354,0.9042,0.01042,0.05411,0.8845,0.02211,0.7147,0.0009856,7410,161
unweighted,gold_upstream_otto4,-0.6068,0.7652,0.4293,0.4326,0.9277,0.5236,0.7763,0.4344,9264,161
without Manaus and Belém,gold_upstream_otto4,-1.15,0.8884,0.1982,0.2144,0.9755,0.2582,0.6559,0.0796,9234,160
2004-2023,gold_upstream_otto4,-3.01,1.043,0.004671,0.0402,1.007,0.00755,0.4968,1.383e-09,12347,161
USD price,gold_upstream_otto4,-3.933,1.755,0.02693,0.06961,1.112,0.003286,1.312,0.002724,9264,161
"never-exploited gold, Otto-4",gold_unexploited_upstream_otto4,-0.8626,1.269,0.498,0.5287,1.618,0.6024,1.14,0.4493,9264,103
"never-exploited gold within 200 km, log(1+count)",log1p_gold_unexploited_upstream_200,-0.1238,0.6939,0.8587,0.8566,0.7104,0.8642,0.3773,0.7429,9264,58



injury_hospitalizations


,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_exposed
spec,exposure,,,,,,,,,,
baseline,gold_upstream_otto4,-0.7521,0.3134,0.01798,0.07001,0.3199,0.03392,0.1444,1.912e-07,9264,161
state x year FE,gold_upstream_otto4,-0.2626,0.2288,0.2533,0.3797,0.217,0.2461,0.07239,0.0002853,9264,161
drop 2015-17,gold_upstream_otto4,-0.7349,0.2974,0.01493,0.06201,0.3066,0.03543,0.1385,1.115e-07,7410,161
unweighted,gold_upstream_otto4,-0.6346,0.2797,0.02512,0.0464,0.2872,0.0443,0.1166,5.299e-08,9264,161
without Manaus and Belém,gold_upstream_otto4,-0.7291,0.3099,0.02031,0.0299,0.3095,0.03361,0.1296,1.824e-08,9234,160
2004-2023,gold_upstream_otto4,-0.5478,0.3154,0.08505,0.1652,0.3139,0.09715,0.09002,1.166e-09,12347,161
USD price,gold_upstream_otto4,-0.9741,0.4713,0.041,0.1142,0.5347,0.08995,0.4035,0.01576,9264,161
"never-exploited gold, Otto-4",gold_unexploited_upstream_otto4,-0.5181,0.3696,0.1636,0.2272,0.3629,0.1753,0.1461,0.0003914,9264,103
"never-exploited gold within 200 km, log(1+count)",log1p_gold_unexploited_upstream_200,-0.2301,0.2026,0.2583,0.2273,0.2251,0.3239,0.1502,0.1254,9264,58


In [5]:
p_columns = ["p_basin", "p_wcb", "p_two_way", "p_pe"]
summary = robustness.assign(significant=lambda d: (d[p_columns] < 0.05).sum(axis=1))
table = summary[summary.exposure.isin([M.EXPOSURE, M.UNEXPLOITED_EXPOSURE]) | summary.exposure.str.startswith("log1p")]
display(table.pivot_table(index="spec", columns="outcome", values="significant", aggfunc="first", sort=False)
        .rename_axis(columns="number of the four p-values < 0.05"))

number of the four p-values < 0.05,anomaly,low_birth_weight,preterm,apgar5_below_6,infant_deaths_nonexternal,infant_deaths_congenital,congenital_hospitalizations,injury_hospitalizations
spec,,,,,,,,
baseline,1,1,1,0,0,4,3,3
state x year FE,1,1,0,0,0,0,0,1
drop 2015-17,1,1,1,0,0,4,3,3
unweighted,0,0,1,0,1,0,0,4
without Manaus and Belém,1,0,0,0,0,4,0,4
2004-2023,1,1,0,0,0,1,4,1
USD price,0,0,0,0,0,4,3,2
"never-exploited gold, Otto-4",1,0,0,0,0,0,0,1
"never-exploited gold within 200 km, log(1+count)",1,0,2,0,0,0,0,0


**Reading the results.** The anomaly coefficient stays between 0.41 and 1.03 and
insignificant under the clustered inferences in every specification; only P-E rejects.
Never-exploited occurrences give 2.36 (Otto-3 p 0.09), close to Moreno-Louzada's number
but on 103 exposed municipalities, and downstream gold, the placebo, gives 1.79 (p 0.11),
as large as upstream gold. The congenital infant-death coefficient survives dropping
2015–17 (0.31), the capitals (0.31), the downstream control (0.32) and the USD price
(0.48), but halves with state×year FE (0.15, p 0.15), falls to 0.07 unweighted and to 0.18
on 2004–2023 (p 0.06), and is insignificant for never-exploited deposits (0.24, p 0.12). The injury placebo is
negative and significant in most specifications, so exposed municipalities' hospital
records trend differently from the rest as the price rises. Read together, the design
picks up differential trends between exposed and unexposed municipalities, not a
price-driven health effect.

## 4. First stage

Does the interaction predict garimpo upstream? The outcomes are the C11 garimpo-gold
share within 50 and 200 km upstream (pp, 2007–2019, before the 2020 reclassification step)
and garimpo malaria cases by infection municipality, summed over the municipalities
intersecting an upstream Otto-4 sub-basin and in the municipality itself (log(1 + cases),
2007–2021). Same sample, FE, weights and inference as the baseline.

In [6]:
first_stage = M.first_stage(frame)
first_stage.to_csv(OUT / "first_stage.csv", index=False)
display(first_stage.set_index(["outcome", "years"])[COLUMNS])

,,estimate,se_basin,p_basin,p_wcb,se_two_way,p_two_way,se_pe,p_pe,n_obs,n_municipalities,n_exposed,n_basins,mean_y
outcome,years,,,,,,,,,,,,,
C11 garimpo gold within 50 km upstream (pp),2007-2019,0.001161,0.0009221,0.2104,0.2667,0.000895,0.2188,0.0002569,6.172e-06,8028,618,161,117,0.008019
C11 garimpo gold within 200 km upstream (pp),2007-2019,0.0009681,0.0007579,0.204,0.2,0.0007377,0.214,0.0001704,1.324e-08,8028,618,161,117,0.005154
"log(1 + garimpo cases upstream, Otto-4)",2007-2021,-0.01071,0.3049,0.972,0.9792,0.3234,0.974,0.1455,0.9413,9264,618,161,117,1.985
log(1 + garimpo cases in the municipality),2007-2021,0.04845,0.1401,0.7301,0.7891,0.1333,0.7217,0.05183,0.3499,9264,618,161,117,0.51


**Reading the results.** There is no first stage. The interaction moves the C11 garimpo-gold
share upstream by 0.001 pp per log point (Otto-3 p 0.20–0.21, on a mean of 0.005–0.008 pp),
and garimpo malaria cases upstream (−0.01, p 0.97) and in the municipality (+0.05,
p 0.73) not at all. Whatever the reduced form picks up, it does not run through measured
garimpo activity upstream.

## 5. Inference

The four standard errors of the baseline anomaly coefficient side by side, and the
random-walk null distribution's implied spread.

In [7]:
anomaly = baseline.set_index("outcome").loc["anomaly"]
data = M.spec_data(frame, "anomaly")
observed, p_rw = M.random_walk_p(data, M.EXPOSURE, M.PRICE)
print(f"anomaly: estimate {anomaly.estimate:.3f}; SE Otto-3 {anomaly.se_basin:.3f}, two-way {anomaly.se_two_way:.3f}, "
      f"P-E {anomaly.se_pe:.3f}; p Otto-3 {anomaly.p_basin:.3f}, WCB {anomaly.p_wcb:.3f}, two-way {anomaly.p_two_way:.3f}, "
      f"P-E {anomaly.p_pe:.3f}, random walk {p_rw:.3f}")
ratios = baseline.assign(pe_over_basin=baseline.se_pe / baseline.se_basin, two_way_over_basin=baseline.se_two_way / baseline.se_basin)
display(ratios.set_index("outcome")[["pe_over_basin", "two_way_over_basin"]])

anomaly: estimate 0.840; SE Otto-3 0.979, two-way 0.964, P-E 0.254; p Otto-3 0.393, WCB 0.426, two-way 0.398, P-E 0.001, random walk 0.786


,pe_over_basin,two_way_over_basin
outcome,,
anomaly,0.2596,0.9847
low_birth_weight,0.4212,0.9932
preterm,0.6466,0.9245
apgar5_below_6,0.8251,1.147
infant_deaths_nonexternal,0.4225,0.8116
infant_deaths_congenital,0.617,0.7724
congenital_hospitalizations,0.8609,1.035
injury_hospitalizations,0.4609,1.021


**Reading the results.** Two-way clustering by basin and year gives SEs within 25% of the
Otto-3 SEs. The P-E SE is 26–86% of the Otto-3 SE; it is close to clustering by year (its
formula sums the score within years), and its normal approximation treats the 15 yearly
prices as independent draws. The gold price is persistent and the residuals are serially
correlated within municipalities, so that assumption fails here, and P-E rejects where
nothing else does (anomaly: P-E p 0.001 against 0.39–0.43). The random-walk
randomization, which keeps the price's persistence, gives p = 0.79 for the anomaly: a
random walk with the price's volatility produces coefficients as large as the observed
one most of the time. Inference in this design should rest on the Otto-3 cluster SEs and
the bootstrap, and the randomization p-values are the conservative bound.